In [ ]:
# Célula 1 — testa o ambiente antes de qualquer processamento
!pip -q install essentia-tensorflow

import sys, platform, re, requests
import essentia, essentia.standard as es
print("Python", sys.version.split()[0], "| máquina", platform.machine(), "| essentia", essentia.__version__)

# 1) O Deezer responde a partir dos servidores do Google?
r = requests.get("https://api.deezer.com/track/3135556", timeout=30).json()
print("Deezer:", r.get("title"), "—", r.get("artist", {}).get("name"), "| tem prévia:", bool(r.get("preview")))

# 2) Quais modelos EmoMusic existem no site oficial da Essentia?
url = "https://essentia.upf.edu/models/classification-heads/emomusic/"
resp = requests.get(url, timeout=30)
print("\nPasta de modelos EmoMusic — status", resp.status_code)
for nome in sorted(set(re.findall(r'href="(emomusic[^"]+)"', resp.text))):
    print(" ", nome)

In [ ]:
# Célula 2 — baixa os modelos, lê as descrições oficiais e testa com 1 prévia
import json, os, requests, numpy as np

BASE = "https://essentia.upf.edu/models"
MODELOS = {
    "musicnn": {"extrator": f"{BASE}/feature-extractors/musicnn/msd-musicnn-1",
                "cabeca":   f"{BASE}/classification-heads/emomusic/emomusic-msd-musicnn-2"},
    "vggish":  {"extrator": f"{BASE}/feature-extractors/vggish/audioset-vggish-3",
                "cabeca":   f"{BASE}/classification-heads/emomusic/emomusic-audioset-vggish-2"},
}
os.makedirs("modelos", exist_ok=True)

def baixar(url_base):
    nome = url_base.split("/")[-1]
    for ext in (".pb", ".json"):
        destino = f"modelos/{nome}{ext}"
        if not os.path.exists(destino):
            r = requests.get(url_base + ext, timeout=120)
            r.raise_for_status()
            open(destino, "wb").write(r.content)
    return f"modelos/{nome}"

info = {}
for nome, m in MODELOS.items():
    ext, cab = baixar(m["extrator"]), baixar(m["cabeca"])
    je, jc = json.load(open(ext + ".json")), json.load(open(cab + ".json"))
    saidas_emb = [o["name"] for o in je["schema"]["outputs"] if o.get("output_purpose") == "embeddings"]
    if not saidas_emb:
        print(nome, "— não achei a saída de embeddings; descrição:", je["schema"]["outputs"])
        continue
    info[nome] = dict(ext=ext, cab=cab, saida_emb=saidas_emb[0],
                      entrada_cab=jc["schema"]["inputs"][0]["name"],
                      saida_cab=jc["schema"]["outputs"][0]["name"],
                      classes=jc.get("classes"))
    print(f"{nome}: ordem das notas = {jc.get('classes')} | "
          f"embeddings = '{saidas_emb[0]}' | cabeça: '{info[nome]['entrada_cab']}' → '{info[nome]['saida_cab']}'")

# teste com uma prévia
prev = requests.get("https://api.deezer.com/track/3135556", timeout=30).json()["preview"]
open("teste.mp3", "wb").write(requests.get(prev, timeout=60).content)
audio = es.MonoLoader(filename="teste.mp3", sampleRate=16000, resampleQuality=4)()
print(f"\nprévia de teste: {len(audio)/16000:.1f} s")

for nome, i in info.items():
    Extrator = es.TensorflowPredictMusiCNN if nome == "musicnn" else es.TensorflowPredictVGGish
    emb = Extrator(graphFilename=i["ext"] + ".pb", output=i["saida_emb"])(audio)
    pred = es.TensorflowPredict2D(graphFilename=i["cab"] + ".pb",
                                  input=i["entrada_cab"], output=i["saida_cab"])(emb)
    medias = dict(zip(i["classes"], pred.mean(axis=0).round(2)))
    print(f"{nome}: embeddings {emb.shape} | notas por pedaço {pred.shape} | média {medias}")

In [ ]:
# Célula 3 — conecta o Google Drive e baixa a PMEmo (base de validação)
from google.colab import drive
drive.mount("/content/drive")

import os
# Pastas no Google Drive: os únicos caminhos do notebook (o que cada uma precisa ter: data/README.md)
PASTA = "/content/drive/MyDrive/playlist-foto"          # audio_teto100.csv entra; notas e embeddings saem
PASTA_PROJETO = "/content/drive/MyDrive/mapa-sonoro"    # base_filtrada.csv e audio_previas/ (célula de dançabilidade)
os.makedirs(PASTA, exist_ok=True)
print("arquivos na pasta do Drive:", os.listdir(PASTA))

!pip -q install gdown
!gdown --folder "https://drive.google.com/drive/folders/1NhN4KaLQPFg9nRNOwne-Lnkxi3nlJHR3" -O pmemo
!cd pmemo && find . -name "*.zip" -exec unzip -q -o {} \;
!find pmemo -maxdepth 3 | head -60
!du -sh pmemo

In [ ]:
# Célula 4 — validação na PMEmo: MusiCNN x VGGish, com e sem normalização de volume
import glob, os, time, numpy as np, pandas as pd
from scipy.stats import pearsonr, spearmanr

# 1) notas gerais (estáticas) da PMEmo: valência e energia vêm em arquivos separados
pasta_pmemo = glob.glob("pmemo/**/V_static.csv", recursive=True)[0].rsplit("/", 1)[0]
assert os.path.exists(f"{pasta_pmemo}/A_static.csv"), "Não achei A_static.csv — me avise."
v = pd.read_csv(f"{pasta_pmemo}/V_static.csv")
a = pd.read_csv(f"{pasta_pmemo}/A_static.csv")
print("colunas V:", list(v.columns), "| colunas A:", list(a.columns))
ann = v.merge(a, on="musicId")
col_v, col_a = "mean_V", "mean_A"
print(f"faixa valência {ann[col_v].min():.2f}–{ann[col_v].max():.2f} | "
      f"faixa energia {ann[col_a].min():.2f}–{ann[col_a].max():.2f}")

pasta_chorus = glob.glob("pmemo/**/Chorus", recursive=True)[0]
ann["arquivo"] = ann["musicId"].map(lambda i: f"{pasta_chorus}/{i}.mp3")
ann = ann[ann["arquivo"].map(os.path.exists)].reset_index(drop=True)
print("refrões com nota e áudio:", len(ann))

# 2) carrega os modelos uma vez
EXTRATORES = {
    "musicnn": es.TensorflowPredictMusiCNN(graphFilename=info["musicnn"]["ext"] + ".pb", output=info["musicnn"]["saida_emb"]),
    "vggish":  es.TensorflowPredictVGGish(graphFilename=info["vggish"]["ext"] + ".pb", output=info["vggish"]["saida_emb"]),
}
CABECAS = {n: es.TensorflowPredict2D(graphFilename=info[n]["cab"] + ".pb",
                                     input=info[n]["entrada_cab"], output=info[n]["saida_cab"]) for n in info}

def ganho_ebu(arquivo, alvo=-23.0):
    """Ganho para levar o volume integrado (EBU R128) a -23 LUFS."""
    estereo, sr = es.AudioLoader(filename=arquivo)()[:2]
    integrado = es.LoudnessEBUR128(sampleRate=sr)(estereo)[2]
    return 10 ** ((alvo - integrado) / 20), integrado

# 3) roda tudo — salva a cada 20 refrões no Drive e retoma de onde parou
parcial = f"{PASTA}/pmemo_predicoes_parcial.csv"
feitos = pd.read_csv(parcial) if os.path.exists(parcial) else pd.DataFrame()
ja = set(feitos["musicId"]) if len(feitos) else set()
linhas = feitos.to_dict("records")
fila = ann[~ann["musicId"].isin(ja)]
print(f"já feitos: {len(ja)} | nesta rodada: {len(fila)}")

t0 = time.time()
for k, (_, r) in enumerate(fila.iterrows(), 1):
    audio = es.MonoLoader(filename=r.arquivo, sampleRate=16000, resampleQuality=4)()
    g, lufs = ganho_ebu(r.arquivo)
    linha = dict(musicId=r.musicId, humano_v=r[col_v], humano_a=r[col_a], lufs=lufs)
    for normalizado, sinal in (("bruto", audio), ("norm", (audio * g).astype(np.float32))):
        for n in EXTRATORES:
            pred = CABECAS[n](EXTRATORES[n](sinal)).mean(axis=0)
            linha[f"{n}_{normalizado}_v"], linha[f"{n}_{normalizado}_a"] = float(pred[0]), float(pred[1])
    linhas.append(linha)
    if k % 20 == 0 or k == len(fila):
        pd.DataFrame(linhas).to_csv(parcial, index=False)
        seg = (time.time() - t0) / k
        print(f"{len(linhas)}/{len(ann)} — {seg:.1f} s por refrão — faltam ~{seg*(len(fila)-k)/60:.0f} min")

res = pd.DataFrame(linhas)
res.to_csv(f"{PASTA}/pmemo_predicoes.csv", index=False)

# 4) métricas que não dependem da escala
def acerto_pares(h, p):
    dh, dp = np.subtract.outer(h, h), np.subtract.outer(p, p)
    i = np.triu_indices(len(h), 1)
    dh, dp = dh[i], dp[i]
    ok = dh != 0
    return float((np.sign(dh[ok]) == np.sign(dp[ok])).mean())

tabela = []
for n in EXTRATORES:
    for normalizado in ("bruto", "norm"):
        for eixo, hum in (("valência", "humano_v"), ("energia", "humano_a")):
            p = res[f"{n}_{normalizado}_{eixo[0] if eixo=='valência' else 'a'}"] if False else res[f"{n}_{normalizado}_{'v' if eixo=='valência' else 'a'}"]
            tabela.append(dict(modelo=n, volume=normalizado, eixo=eixo,
                               pearson=round(pearsonr(res[hum], p)[0], 3),
                               spearman=round(spearmanr(res[hum], p)[0], 3),
                               acerto_pares=round(acerto_pares(res[hum].values, p.values), 3)))
tabela = pd.DataFrame(tabela)
tabela.to_csv(f"{PASTA}/pmemo_metricas.csv", index=False)
print(f"\ntempo total: {(time.time()-t0)/60:.1f} min | {(time.time()-t0)/len(ann):.2f} s por refrão")
print(tabela.to_string(index=False))

In [ ]:
# Célula 5 — escolha do modelo: 200 refrões sorteados, cortados em 30 s (como as prévias)
import glob, os, time, numpy as np, pandas as pd
from scipy.stats import spearmanr

pasta_pmemo = glob.glob("pmemo/**/V_static.csv", recursive=True)[0].rsplit("/", 1)[0]
ann = pd.read_csv(f"{pasta_pmemo}/V_static.csv").merge(
      pd.read_csv(f"{pasta_pmemo}/A_static.csv"), on="musicId")
ann["arquivo"] = ann["musicId"].map(lambda i: f"{pasta_pmemo}/Chorus/{i}.mp3")
ann = ann[ann["arquivo"].map(os.path.exists)]
amostra = ann.sample(200, random_state=42).reset_index(drop=True)

CONFIGS = {
    "musicnn_padrao": es.TensorflowPredictMusiCNN(graphFilename=info["musicnn"]["ext"] + ".pb",
                                                  output=info["musicnn"]["saida_emb"]),
    "musicnn_rapido": es.TensorflowPredictMusiCNN(graphFilename=info["musicnn"]["ext"] + ".pb",
                                                  output=info["musicnn"]["saida_emb"], patchHopSize=187),
    "vggish":         es.TensorflowPredictVGGish(graphFilename=info["vggish"]["ext"] + ".pb",
                                                 output=info["vggish"]["saida_emb"]),
}
CAB = {n: es.TensorflowPredict2D(graphFilename=info[n]["cab"] + ".pb",
                                 input=info[n]["entrada_cab"], output=info[n]["saida_cab"]) for n in info}

def corte30(audio, sr=16000):
    n = 30 * sr
    if len(audio) <= n:
        return audio
    ini = (len(audio) - n) // 2
    return audio[ini:ini + n]

linhas, tempos, t0 = [], {c: 0.0 for c in CONFIGS}, time.time()
for k, r in amostra.iterrows():
    audio = corte30(es.MonoLoader(filename=r.arquivo, sampleRate=16000, resampleQuality=4)())
    linha = dict(musicId=r.musicId, humano_v=r.mean_V, humano_a=r.mean_A)
    for c, extrator in CONFIGS.items():
        t = time.time()
        pred = CAB[c.split("_")[0]](extrator(audio)).mean(axis=0)
        tempos[c] += time.time() - t
        linha[f"{c}_v"], linha[f"{c}_a"] = float(pred[0]), float(pred[1])
    linhas.append(linha)
    if (k + 1) % 20 == 0:
        print(f"{k+1}/200 — faltam ~{(time.time()-t0)/(k+1)*(200-k-1)/60:.0f} min")

res = pd.DataFrame(linhas)
res.to_csv(f"{PASTA}/escolha_modelo_200.csv", index=False)

def acerto_pares(h, p):
    i = np.triu_indices(len(h), 1)
    dh, dp = np.subtract.outer(h, h)[i], np.subtract.outer(p, p)[i]
    ok = dh != 0
    return float((np.sign(dh[ok]) == np.sign(dp[ok])).mean())

tab = []
for c in CONFIGS:
    for eixo, h in (("valência", "humano_v"), ("energia", "humano_a")):
        p = res[f"{c}_{'v' if eixo == 'valência' else 'a'}"]
        tab.append(dict(config=c, eixo=eixo,
                        spearman=round(spearmanr(res[h], p)[0], 3),
                        acerto_pares=round(acerto_pares(res[h].values, p.values), 3)))
tab = pd.DataFrame(tab)
tab.to_csv(f"{PASTA}/escolha_modelo_metricas.csv", index=False)
print(tab.to_string(index=False))
print("\nhoras estimadas para as 12.329 prévias:",
      {c: round(tempos[c] / len(res) * 12329 / 3600, 1) for c in CONFIGS})

In [ ]:
# Célula 6 — dois ajustes no vencedor (musicnn_rapido), nos mesmos 200 refrões
def ganho_ebu(arquivo, alvo=-23.0):
    estereo, sr = es.AudioLoader(filename=arquivo)()[:2]
    integrado = es.LoudnessEBUR128(sampleRate=sr)(estereo)[2]
    return 10 ** ((alvo - integrado) / 20)

def corte(audio, seg, sr=16000):
    n = seg * sr
    if len(audio) <= n:
        return audio
    ini = (len(audio) - n) // 2
    return audio[ini:ini + n]

ext, cab = CONFIGS["musicnn_rapido"], CAB["musicnn"]
linhas, tempo = [], {"norm30": 0.0, "bruto15": 0.0}
for k, r in amostra.iterrows():
    bruto = es.MonoLoader(filename=r.arquivo, sampleRate=16000, resampleQuality=4)()
    t0 = time.time()
    g = ganho_ebu(r.arquivo)
    p_norm = cab(ext((corte(bruto, 30) * g).astype(np.float32))).mean(axis=0)
    tempo["norm30"] += time.time() - t0
    t0 = time.time()
    p15 = cab(ext(corte(bruto, 15))).mean(axis=0)
    tempo["bruto15"] += time.time() - t0
    linhas.append(dict(musicId=r.musicId, norm30_v=float(p_norm[0]), norm30_a=float(p_norm[1]),
                       bruto15_v=float(p15[0]), bruto15_a=float(p15[1])))
    if (k + 1) % 50 == 0:
        print(f"{k+1}/200")

aj = res[["musicId", "humano_v", "humano_a", "musicnn_rapido_v", "musicnn_rapido_a"]].merge(
     pd.DataFrame(linhas), on="musicId")
tab2 = []
for nome, col in (("rapido 30 s, volume original", "musicnn_rapido"),
                  ("rapido 30 s, volume normalizado", "norm30"),
                  ("rapido 15 s, volume original", "bruto15")):
    for eixo, h, s in (("valência", "humano_v", "v"), ("energia", "humano_a", "a")):
        p = aj[f"{col}_{s}"]
        tab2.append(dict(config=nome, eixo=eixo,
                         spearman=round(spearmanr(aj[h], p)[0], 3),
                         acerto_pares=round(acerto_pares(aj[h].values, p.values), 3)))
tab2 = pd.DataFrame(tab2)
tab2.to_csv(f"{PASTA}/ajustes_musicnn_rapido.csv", index=False)
print(tab2.to_string(index=False))
print("\nhoras estimadas para as 12.329:", {c: round(v / len(aj) * 12329 / 3600, 1) for c, v in tempo.items()})

## Célula 6b — cabeça DEAM × cabeça emoMusic (review, item "newer audio model")

O review (`docs/REVIEW.md`, seção 1, *The song scores rest on a weak base model*) sugere: *"Use a newer audio model: Essentia's DEAM heads, or a CLAP or MERT encoder."*

Este teste troca **só o "cérebro"**: o ouvido e a configuração final ficam iguais (msd-musicnn-1 com `patchHopSize=187`, 15 s do meio, volume original). As duas cabeças são aplicadas ao **mesmo** embedding de cada refrão, então a comparação é pareada.

- **Base:** PMEmo 2018, os 767 refrões com nota estática. Nenhuma das duas cabeças foi treinada nela.
- **Número principal:** os 567 refrões que não participaram da escolha do modelo (os 200 da Célula 5 ficam de fora, mesma semente 42).
- **Medidas:** correlação de Spearman e acerto em pares (as mesmas da escolha do modelo). Conferência: o emoMusic deve reproduzir os números da prova completa (valência 0,491 / 0,674; energia 0,656 / 0,740).
- **Diferença:** Spearman(DEAM) − Spearman(emoMusic) nos 567, com IC 95% por bootstrap pareado (10.000 reamostras dos refrões).

**Regra de decisão (fixada antes de rodar):** o DEAM substitui o emoMusic se, nos 567, o IC 95% da diferença de Spearman ficar inteiro acima de zero em pelo menos um eixo **e** o do outro eixo não ficar inteiro abaixo de zero. Caso contrário, o emoMusic continua e o resultado é registrado em `docs/decisoes.md`.

Precisa das Células 1, 2 e 3 (Essentia, modelos e PMEmo). Salva no Drive: `pmemo_deam_vs_emomusic.csv` (notas por refrão; retoma se cair) e `pmemo_deam_vs_emomusic_metricas.csv`.


In [ ]:
# Célula 6b — cabeça DEAM x cabeça emoMusic, mesmo ouvido (msd-musicnn-1, patchHopSize=187, 15 s do meio)
import glob, json, os, time, numpy as np, pandas as pd
from scipy.stats import spearmanr

# 1) cabeça DEAM; nomes de entrada/saída e ordem das notas vêm do .json oficial
cab_deam = baixar(f"{BASE}/classification-heads/deam/deam-msd-musicnn-2")
jd = json.load(open(cab_deam + ".json"))
CLASSES = {"emomusic": info["musicnn"]["classes"], "deam": jd.get("classes")}
print("ordem das notas:", CLASSES)
for nome, cl in CLASSES.items():
    if [str(c).lower() for c in (cl or [])] != ["valence", "arousal"]:
        raise ValueError(f"Ordem das notas inesperada em {nome}: {cl}. Pare e me avise.")

extrator = es.TensorflowPredictMusiCNN(graphFilename=info["musicnn"]["ext"] + ".pb",
                                       output=info["musicnn"]["saida_emb"], patchHopSize=187)
CABECAS = {
    "emomusic": es.TensorflowPredict2D(graphFilename=info["musicnn"]["cab"] + ".pb",
                                       input=info["musicnn"]["entrada_cab"], output=info["musicnn"]["saida_cab"]),
    "deam":     es.TensorflowPredict2D(graphFilename=cab_deam + ".pb",
                                       input=jd["schema"]["inputs"][0]["name"], output=jd["schema"]["outputs"][0]["name"]),
}

# 2) PMEmo: as mesmas notas estáticas e os mesmos 200 da escolha do modelo (Célula 5, semente 42)
pasta_pmemo = glob.glob("pmemo/**/V_static.csv", recursive=True)[0].rsplit("/", 1)[0]
ann = pd.read_csv(f"{pasta_pmemo}/V_static.csv").merge(pd.read_csv(f"{pasta_pmemo}/A_static.csv"), on="musicId")
ann["arquivo"] = ann["musicId"].map(lambda i: f"{pasta_pmemo}/Chorus/{i}.mp3")
ann = ann[ann["arquivo"].map(os.path.exists)]
escolha = set(ann.sample(200, random_state=42)["musicId"])
ann = ann.reset_index(drop=True)
print(f"refrões: {len(ann)} | fora da escolha: {(~ann.musicId.isin(escolha)).sum()}")

def corte(audio, seg, sr=16000):
    n = seg * sr
    if len(audio) <= n:
        return audio
    ini = (len(audio) - n) // 2
    return audio[ini:ini + n]

# 3) notas das duas cabeças no mesmo embedding; salva a cada 50 refrões e retoma se cair
PARCIAL = f"{PASTA}/pmemo_deam_vs_emomusic.csv"
feitos = pd.read_csv(PARCIAL) if os.path.exists(PARCIAL) else pd.DataFrame(columns=["musicId"])
linhas = feitos.to_dict("records")
fila = ann[~ann.musicId.isin(set(feitos.musicId))]
print(f"já feitos: {len(feitos)} | nesta rodada: {len(fila)}")
t0 = time.time()
for k, r in enumerate(fila.itertuples(index=False), 1):
    audio = es.MonoLoader(filename=r.arquivo, sampleRate=16000, resampleQuality=4)()
    emb = extrator(corte(audio, 15))
    linha = dict(musicId=r.musicId, humano_v=r.mean_V, humano_a=r.mean_A, escolha=r.musicId in escolha)
    for nome, cab in CABECAS.items():
        p = cab(emb).mean(axis=0)
        linha[f"{nome}_v"], linha[f"{nome}_a"] = float(p[0]), float(p[1])
    linhas.append(linha)
    if k % 50 == 0 or k == len(fila):
        pd.DataFrame(linhas).to_csv(PARCIAL, index=False)
        print(f"{k}/{len(fila)} — faltam ~{(time.time() - t0) / k * (len(fila) - k) / 60:.0f} min")
res = pd.DataFrame(linhas)
res["escolha"] = res["escolha"].astype(str).str.lower().eq("true")

# 4) medidas
def acerto_pares(h, p):
    i = np.triu_indices(len(h), 1)
    dh, dp = np.subtract.outer(h, h)[i], np.subtract.outer(p, p)[i]
    ok = dh != 0
    return float((np.sign(dh[ok]) == np.sign(dp[ok])).mean())

EIXOS = (("valência", "humano_v", "v"), ("energia", "humano_a", "a"))
tab = []
for grupo, df in (("567 fora da escolha", res[~res.escolha]), ("todos os 767", res)):
    for eixo, h, s in EIXOS:
        for nome in CABECAS:
            tab.append(dict(grupo=grupo, eixo=eixo, cabeca=nome, n=len(df),
                            spearman=round(spearmanr(df[h], df[f"{nome}_{s}"])[0], 3),
                            acerto_pares=round(acerto_pares(df[h].values, df[f"{nome}_{s}"].values), 3)))
tab = pd.DataFrame(tab)
tab.to_csv(f"{PASTA}/pmemo_deam_vs_emomusic_metricas.csv", index=False)
print(tab.to_string(index=False))
print("\nconferência (prova completa, 567): emoMusic valência 0,491 / 0,674 | energia 0,656 / 0,740")

# 5) diferença DEAM − emoMusic nos 567, IC 95% por bootstrap pareado
fora = res[~res.escolha].reset_index(drop=True)
rng = np.random.default_rng(0)
ic = {}
for eixo, h, s in EIXOS:
    hv, ev, dv = fora[h].values, fora[f"emomusic_{s}"].values, fora[f"deam_{s}"].values
    obs = spearmanr(hv, dv)[0] - spearmanr(hv, ev)[0]
    difs = []
    for _ in range(10_000):
        i = rng.integers(len(fora), size=len(fora))
        difs.append(spearmanr(hv[i], dv[i])[0] - spearmanr(hv[i], ev[i])[0])
    lo, hi = np.quantile(difs, [0.025, 0.975])
    ic[eixo] = (lo, hi)
    print(f"{eixo}: DEAM − emoMusic = {obs:+.3f}  IC 95% [{lo:+.3f}; {hi:+.3f}]")

# 6) regra de decisão (fixada antes de rodar)
acima = [e for e, (lo, hi) in ic.items() if lo > 0]
abaixo = [e for e, (lo, hi) in ic.items() if hi < 0]
if acima and not abaixo:
    print(f"\nDECISÃO: o DEAM substitui o emoMusic (melhor em {', '.join(acima)}, sem piora em nenhum eixo).")
else:
    print("\nDECISÃO: o emoMusic continua" + (f" (DEAM pior em {', '.join(abaixo)})." if abaixo else " (sem melhora clara do DEAM)."))


In [ ]:
# Célula 7 — notas das 12.329 prévias: MusiCNN rápido + EmoMusic, trecho do meio, volume original
import os, time, json, threading, tempfile, datetime, numpy as np, pandas as pd, requests
from concurrent.futures import ThreadPoolExecutor

SEGUNDOS = 15            # 15 = ~2,4 h | 30 = ~4,3 h (decisão da usuária)
LOTE = 200

CSV_ENTRADA = f"{PASTA}/audio_teto100.csv"
SAIDA = f"{PASTA}/notas_musicas.csv"
PASTA_EMB = f"{PASTA}/embeddings"
os.makedirs(PASTA_EMB, exist_ok=True)

musicas = pd.read_csv(CSV_ENTRADA)
musicas = musicas[musicas["status"] == "ok"][["id", "deezer_id"]].copy()
musicas["deezer_id"] = musicas["deezer_id"].astype(int)
feitos = set(pd.read_csv(SAIDA)["id"]) if os.path.exists(SAIDA) else set()
fila = musicas[~musicas["id"].isin(feitos)].reset_index(drop=True)
print(f"total {len(musicas)} | já feitas {len(feitos)} | nesta rodada {len(fila)} | trecho {SEGUNDOS} s")

extrator = es.TensorflowPredictMusiCNN(graphFilename=info["musicnn"]["ext"] + ".pb",
                                       output=info["musicnn"]["saida_emb"], patchHopSize=187)
cabeca = es.TensorflowPredict2D(graphFilename=info["musicnn"]["cab"] + ".pb",
                                input=info["musicnn"]["entrada_cab"], output=info["musicnn"]["saida_cab"])

json.dump(dict(essentia=essentia.__version__, extrator="msd-musicnn-1 (patchHopSize=187)",
               cabeca="emomusic-msd-musicnn-2", trecho=f"{SEGUNDOS} s do meio", volume="original",
               validacao_pmemo_200="ver escolha_modelo_metricas.csv e ajustes_musicnn_rapido.csv"),
          open(f"{PASTA}/notas_musicas_config.json", "w"), ensure_ascii=False, indent=2)

_trava, _ultimo = threading.Lock(), [0.0]
def esperar_vez(intervalo=0.125):                    # no máximo 8 pedidos/s à API do Deezer
    with _trava:
        espera = _ultimo[0] + intervalo - time.monotonic()
        if espera > 0:
            time.sleep(espera)
        _ultimo[0] = time.monotonic()

def baixar_previa(deezer_id):
    """Pede um link novo ao Deezer (os links expiram) e baixa a prévia."""
    for tentativa in range(4):
        try:
            esperar_vez()
            d = requests.get(f"https://api.deezer.com/track/{deezer_id}", timeout=30).json()
            if "error" in d:
                if d["error"].get("code") == 4:
                    time.sleep(5 * (tentativa + 1))
                    continue
                return None, "erro_deezer"
            if not d.get("preview"):
                return None, "sem_preview"
            r = requests.get(d["preview"], timeout=60)
            if r.status_code == 200 and len(r.content) > 10_000:
                return r.content, "ok"
        except (requests.RequestException, ValueError):
            pass
        time.sleep(3 * (tentativa + 1))
    return None, "falha_download"

def corte(audio, seg, sr=16000):
    n = seg * sr
    if len(audio) <= n:
        return audio
    ini = (len(audio) - n) // 2
    return audio[ini:ini + n]

lotes = [fila.iloc[i:i + LOTE] for i in range(0, len(fila), LOTE)]
novo, t0, feitas = not os.path.exists(SAIDA), time.time(), 0
with ThreadPoolExecutor(max_workers=4) as ex:
    proximo = ex.map(baixar_previa, lotes[0]["deezer_id"]) if lotes else None
    for k, lote in enumerate(lotes):
        baixados = list(proximo)
        if k + 1 < len(lotes):                       # já começa a baixar o próximo lote
            proximo = ex.map(baixar_previa, lotes[k + 1]["deezer_id"])
        linhas, ids_emb, embs = [], [], []
        for (_, m), (conteudo, status) in zip(lote.iterrows(), baixados):
            linha = dict(id=m.id, deezer_id=m.deezer_id, status=status,
                         processado_em=datetime.date.today().isoformat())
            if conteudo:
                with tempfile.NamedTemporaryFile(suffix=".mp3") as tmp:
                    tmp.write(conteudo)
                    tmp.flush()
                    audio = es.MonoLoader(filename=tmp.name, sampleRate=16000, resampleQuality=4)()
                trecho = corte(audio, SEGUNDOS)
                if len(trecho) < 3 * 16000:
                    linha["status"] = "audio_curto"
                else:
                    emb = extrator(trecho)
                    pred = cabeca(emb).mean(axis=0)
                    linha.update(valencia=float(pred[0]), energia=float(pred[1]),
                                 n_pedacos=len(emb), duracao_previa=round(len(audio) / 16000, 1))
                    ids_emb.append(m.id)
                    embs.append(emb.mean(axis=0))
            linhas.append(linha)
        pd.DataFrame(linhas).to_csv(SAIDA, mode="a", header=novo, index=False)
        novo = False
        if embs:
            np.savez_compressed(f"{PASTA_EMB}/lote_{int(time.time())}.npz",
                                ids=np.array(ids_emb), emb=np.stack(embs))
        feitas += len(lote)
        seg = (time.time() - t0) / feitas
        print(f"{len(feitos) + feitas}/{len(musicas)} — {seg:.2f} s por música — "
              f"faltam ~{seg * (len(fila) - feitas) / 60:.0f} min")

res_final = pd.read_csv(SAIDA)
print("\n=== Resumo ===")
print(res_final["status"].value_counts().to_string())

## Célula 8 — dançabilidade das 2.812 músicas do site (review: "learn danceability from the audio")

O review (`docs/REVIEW.md`, seção 1, *Medium: party mode is a hand-written rule with five magic constants*): *"The notebook already names the right fix: learn danceability from the audio."*

- **Modelo:** `danceability-msd-musicnn-1` da Essentia, uma cabeça sobre o **mesmo ouvido** das notas (msd-musicnn-1). Saída: probabilidade de `danceable` e `not_danceable` (ordem lida do `.json` oficial). Validação publicada pela Essentia: acurácia normalizada de 0,95 em validação cruzada de 5 folds, numa coleção interna do MTG de 306 faixas inteiras.
- **Mesma configuração das notas:** `patchHopSize=187`, 15 s do meio da prévia, volume original. A cabeça é aplicada a cada pedaço e as probabilidades são somadas pela média (o mesmo procedimento das notas de humor e energia).
- **Músicas:** só as 2.812 que estão no site (`site/api/_dados/musicas.json`, lido do GitHub). Cada prévia é baixada de novo do Deezer pelo `deezer_id`.
- **Saída no Drive:** `dancabilidade.csv` (`deezer_id`, `dancante` = probabilidade média de *danceable*, `n_pedacos`, `status`). Salva a cada 200 músicas e retoma se cair.

Precisa das Células 1, 2 e 3. Esta célula substitui uma versão antiga que lia as prévias de `audio_previas/` no Drive (não estão lá) e não usava a configuração das notas.
Como a dançabilidade entra no modo festa é decidido depois, com os números na mão.


In [ ]:
# Célula 8 — dançabilidade das 2.812 músicas do site, mesmo ouvido e mesma configuração das notas
import json, os, time, tempfile, threading, numpy as np, pandas as pd, requests
from concurrent.futures import ThreadPoolExecutor

# 1) modelo: nomes e ordem das classes vêm do .json oficial
cab_danca = baixar(f"{BASE}/classification-heads/danceability/danceability-msd-musicnn-1")
jdanca = json.load(open(cab_danca + ".json"))
classes = [str(c).lower() for c in jdanca["classes"]]
if "danceable" not in classes:
    raise ValueError(f"Classes inesperadas: {classes}. Pare e me avise.")
I_DANCA = classes.index("danceable")
saida_danca = next(o["name"] for o in jdanca["schema"]["outputs"] if o.get("output_purpose") == "predictions")
extrator = es.TensorflowPredictMusiCNN(graphFilename=info["musicnn"]["ext"] + ".pb",
                                       output=info["musicnn"]["saida_emb"], patchHopSize=187)
cabeca_danca = es.TensorflowPredict2D(graphFilename=cab_danca + ".pb",
                                      input=jdanca["schema"]["inputs"][0]["name"], output=saida_danca)
print("classes:", classes, "| saída:", saida_danca)

# 2) as músicas do site
URL_MUSICAS = "https://raw.githubusercontent.com/SofiaLimar/Projeto-musical-Vibez/main/site/api/_dados/musicas.json"
site = pd.DataFrame(requests.get(URL_MUSICAS, timeout=60).json())[["deezer_id"]].drop_duplicates()
SAIDA = f"{PASTA}/dancabilidade.csv"
feitos = set(pd.read_csv(SAIDA)["deezer_id"]) if os.path.exists(SAIDA) else set()
fila = site[~site.deezer_id.isin(feitos)].reset_index(drop=True)
print(f"músicas do site: {len(site)} | já feitas: {len(feitos)} | nesta rodada: {len(fila)}")

# 3) download das prévias (o mesmo cuidado da Célula 7: no máximo 8 pedidos/s, links novos, novas tentativas)
_trava, _ultimo = threading.Lock(), [0.0]
def esperar_vez(intervalo=0.125):
    with _trava:
        espera = _ultimo[0] + intervalo - time.monotonic()
        if espera > 0:
            time.sleep(espera)
        _ultimo[0] = time.monotonic()

def baixar_previa(deezer_id):
    for tentativa in range(4):
        try:
            esperar_vez()
            d = requests.get(f"https://api.deezer.com/track/{deezer_id}", timeout=30).json()
            if "error" in d:
                if d["error"].get("code") == 4:
                    time.sleep(5 * (tentativa + 1))
                    continue
                return None, "erro_deezer"
            if not d.get("preview"):
                return None, "sem_preview"
            r = requests.get(d["preview"], timeout=60)
            if r.status_code == 200 and len(r.content) > 10_000:
                return r.content, "ok"
        except (requests.RequestException, ValueError):
            pass
        time.sleep(3 * (tentativa + 1))
    return None, "falha_download"

def corte(audio, seg, sr=16000):
    n = seg * sr
    if len(audio) <= n:
        return audio
    ini = (len(audio) - n) // 2
    return audio[ini:ini + n]

# 4) processa em lotes de 200, baixando o próximo lote enquanto calcula o atual
LOTE = 200
lotes = [fila.iloc[i:i + LOTE] for i in range(0, len(fila), LOTE)]
novo, t0, feitas = not os.path.exists(SAIDA), time.time(), 0
with ThreadPoolExecutor(max_workers=4) as ex:
    proximo = ex.map(baixar_previa, lotes[0]["deezer_id"]) if lotes else None
    for k, lote in enumerate(lotes):
        baixados = list(proximo)
        if k + 1 < len(lotes):
            proximo = ex.map(baixar_previa, lotes[k + 1]["deezer_id"])
        linhas = []
        for did, (conteudo, status) in zip(lote["deezer_id"], baixados):
            linha = dict(deezer_id=did, dancante=np.nan, n_pedacos=0, status=status)
            if conteudo:
                with tempfile.NamedTemporaryFile(suffix=".mp3") as tmp:
                    tmp.write(conteudo)
                    tmp.flush()
                    audio = es.MonoLoader(filename=tmp.name, sampleRate=16000, resampleQuality=4)()
                trecho = corte(audio, 15)
                if len(trecho) < 3 * 16000:
                    linha["status"] = "audio_curto"
                else:
                    emb = extrator(trecho)
                    p = cabeca_danca(emb).mean(axis=0)
                    linha.update(dancante=float(p[I_DANCA]), n_pedacos=len(emb))
            linhas.append(linha)
        pd.DataFrame(linhas).to_csv(SAIDA, mode="a", header=novo, index=False)
        novo = False
        feitas += len(lote)
        seg = (time.time() - t0) / feitas
        print(f"{len(feitos) + feitas}/{len(site)} — {seg:.2f} s por música — faltam ~{seg * (len(fila) - feitas) / 60:.0f} min")

res = pd.read_csv(SAIDA)
print("\n=== Resumo ===")
print(res["status"].value_counts().to_string())
print(res["dancante"].describe().round(3).to_string())
